In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ! pip install datasets transformers sacrebleu torch sentencepiece transformers[sentencepiece]
! pip install --upgrade datasets transformers sacrebleu transformers[sentencepiece]
!pip install torch==2.3.0 torchtext==0.18.0 torchaudio==2.3.0 torchvision==0.18.0
# !pip install pyarrow==14.0.1
%pip install evaluate

In [ ]:
# !pip3 uninstall --yes torch torchaudio torchvision torchtext torchdata
# !pip3 install torch torchaudio torchvision torchtext torchdata
# !pip install evaluate
!pip install portalocker

In [ ]:
# !pip install -U spacy
# !python -m spacy download en_core_web_sm

In [ ]:
# !pip install torchtext==0.17.2

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import pandas as pd
import spacy
import datasets
import torchtext
# import evaluate
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torch import Tensor
from torch.nn import Transformer
import math
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchtext.transforms import BERTTokenizer
from transformers import AdamW
import sys
from timeit import default_timer as timer
%matplotlib inline
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from typing import Iterable, List
import copy
from collections import Counter
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from transformers import BertTokenizer

In [ ]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [ ]:
df = pd.read_csv('train.csv')

df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

# df = df.iloc[:600000, :]

# df = df.iloc[:3000, :]

In [ ]:
validation_data = pd.read_csv('val.csv')

test_data = pd.read_csv('test.csv')

In [ ]:
len(df), len(validation_data), len(test_data)

In [ ]:
train_df = df
valid_df = validation_data
test_df = test_data

In [ ]:
train_source_sentences = list(train_df['source sentence'].values)
train_target_sentences = list(train_df['target sentence'].values)

valid_source_sentences = list(valid_df['source sentence'].values)
valid_target_sentences = list(valid_df['target sentence'].values)

test_source_sentences = list(test_df['source sentence'].values)
test_target_sentences = list(test_df['target sentence'].values)

print(f"Train Source: {len(train_source_sentences)}")
print(f"Valid Source: {len(valid_source_sentences)}")
print(f"Test Source: {len(test_source_sentences)}")
print(f"Train Target: {len(train_target_sentences)}")
print(f"Valid Target: {len(valid_target_sentences)}")
print(f"Test Target: {len(test_target_sentences)}")

In [ ]:
max(len(x) for x in train_source_sentences), max(len(x) for x in train_target_sentences), max(len(x) for x in test_source_sentences), max(len(x) for x in test_target_sentences),

In [ ]:
PERCENTILE = 97
print( f"{PERCENTILE}th percentile length in Train Source: {np.percentile([len(x) for x in train_source_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length in Train Target: {np.percentile([len(x) for x in train_target_sentences], PERCENTILE)}" )

print( f"{PERCENTILE}th percentile length in Test Source: {np.percentile([len(x) for x in test_source_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length in Test Target: {np.percentile([len(x) for x in test_target_sentences], PERCENTILE)}" )

In [ ]:
train_data = {'source': train_source_sentences, 'target': train_target_sentences}
valid_data = {'source': valid_source_sentences, 'target': valid_target_sentences}
test_data = {'source': test_source_sentences, 'target': test_target_sentences}

# Convert dictionaries to Hugging Face Dataset
train_dataset = datasets.Dataset.from_dict(train_data)
val_dataset = datasets.Dataset.from_dict(valid_data)
test_dataset = datasets.Dataset.from_dict(test_data)

In [ ]:
val_dataset

In [ ]:
src_nlp = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
tgt_nlp = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

In [ ]:
def tokenize_example(example, tgt_nlp, src_nlp, max_length, lower, sos_token, eos_token):
    tgt_tokens = [token for token in tgt_nlp.tokenize(example["target"])][:max_length]
    src_tokens = [token for token in src_nlp.tokenize(example["source"])][:max_length]
    tgt_tokens = [sos_token] + tgt_tokens + [eos_token]
    src_tokens = [sos_token] + src_tokens + [eos_token]
    return {"tgt_tokens": tgt_tokens, "src_tokens": src_tokens}

In [ ]:
max_length = 1000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

# Dictionary of parameters to pass to the function
fn_kwargs_params = {
    "tgt_nlp": tgt_nlp,
    "src_nlp": src_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# Apply the tokenization function to datasets
train_data = train_dataset.map(tokenize_example, fn_kwargs=fn_kwargs_params)
valid_data = val_dataset.map(tokenize_example, fn_kwargs=fn_kwargs_params)
test_data = test_dataset.map(tokenize_example, fn_kwargs=fn_kwargs_params)

In [ ]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

target_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["tgt_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

source_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["src_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

In [ ]:
assert target_vocab[unk_token] == source_vocab[unk_token]
assert target_vocab[pad_token] == source_vocab[pad_token]

unk_index = target_vocab[unk_token]
pad_index = target_vocab[pad_token]

In [ ]:
target_vocab.set_default_index(unk_index)
source_vocab.set_default_index(unk_index)

In [ ]:
def numericalize_example(example, target_vocab, source_vocab):
    target_ids = target_vocab.lookup_indices(example["tgt_tokens"])
    source_ids = source_vocab.lookup_indices(example["src_tokens"])
    return {"target_ids": target_ids, "source_ids": source_ids}

In [ ]:
fn_kwargs = {"target_vocab": target_vocab, "source_vocab": source_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

In [ ]:
data_type = "torch"
format_columns = ["target_ids", "source_ids"]

train_data = train_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

In [ ]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_target_ids = [example["target_ids"] for example in batch]
        batch_source_ids = [example["source_ids"] for example in batch]
        batch_target_ids = nn.utils.rnn.pad_sequence(batch_target_ids, padding_value=pad_index)
        batch_source_ids = nn.utils.rnn.pad_sequence(batch_source_ids, padding_value=pad_index)
        batch = {
            "target_ids": batch_target_ids,
            "source_ids": batch_source_ids,
        }
        return batch

    return collate_fn

In [ ]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [ ]:
batch_size = 16
valid_batch_size = 64

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, valid_batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self, input_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, dropout
    ):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, encoder_hidden_dim, bidirectional=True)
        self.fc = nn.Linear(encoder_hidden_dim * 2, decoder_hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        # print(f"src len: {src.shape}")
        embedded = self.dropout(self.embedding(src))
        # print(f"embedding layer: {embedded.shape}")
        # embedded = [src length, batch size, embedding dim]
        outputs, hidden = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # hidden is stacked [forward_1, backward_1, forward_2, backward_2, ...]
        # outputs are always from the last layer
        # hidden [-2, :, : ] is the last of the forwards RNN
        # hidden [-1, :, : ] is the last of the backwards RNN
        # initial decoder hidden is final hidden state of the forwards and backwards
        # encoder RNNs fed through a linear layer
        hidden = torch.tanh(
            self.fc(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        )
        # print(f"hidden layer: {hidden.shape}")
        # outputs = [src length, batch size, encoder hidden dim * 2]
        # hidden = [batch size, decoder hidden dim]
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        dropout,
    ):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, decoder_hidden_dim)
        self.fc_out = nn.Linear(decoder_hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden):
        # input = [batch size]
        # hidden = [1, batch size, decoder hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, hidden = self.rnn(embedded, hidden.unsqueeze(0))
        # output = [1, batch size, decoder hidden dim]
        # hidden = [1, batch size, decoder hidden dim]
        # seq len, n layers and n directions will always be 1 in this decoder
        # therefore output == hidden
        output = output.squeeze(0)
        # output = [batch size, decoder hidden dim]
        prediction = self.fc_out(output)
        # prediction = [batch size, output dim]
        return prediction, hidden.squeeze(0)


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        batch_size = src.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)

        # encode the source sequence
        encoder_outputs, hidden = self.encoder(src)

        # first input to the decoder is the <sos> token
        input = trg[0, :]

        for t in range(1, trg_length):
            # insert input token embedding and previous hidden state
            # receive output tensor (predictions) and new hidden state
            output, hidden = self.decoder(input, hidden)
            # output = [batch size, output dim]
            outputs[t] = output
            # decide if we are going to use teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)  # get the highest predicted token
            input = trg[t] if teacher_force else top1
        
        return outputs

In [ ]:
input_dim = len(source_vocab)
output_dim = len(target_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
encoder_hidden_dim = 512
decoder_hidden_dim = 512
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# attention = Attention(encoder_hidden_dim, decoder_hidden_dim)

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    encoder_hidden_dim,
    decoder_hidden_dim,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    encoder_hidden_dim,
    decoder_hidden_dim,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device)

for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

model = model.to(device)

In [ ]:
print(model)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index, label_smoothing=0.1)

In [ ]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["source_ids"].to(device)
            trg = batch["target_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=0.001)
N_EPOCHS = 10
epoch = 0
loss = 10e9
clip = 1.0
teacher_forcing_ratio = 0.5

PATH = '/kaggle/working/PwC_DAE_BERT_SMOOTH_64_noATT_V3.pth'
BEST_PATH = '/kaggle/working/PwC_DAE_BERT_SMOOTH_BESTPATH_64_noATT_V3.pth'
VAL_BEST_PATH = '/kaggle/working/PwC_DAE_BERT_SMOOTH_VAL_BEST_PATH_noATT_64_V3.pth'

In [ ]:
print("incorporating model checkpoint")

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated model checkpoint")

In [ ]:
print("incorporating best model checkpoint")

if os.path.exists(BEST_PATH):
    checkpoint = torch.load(BEST_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated best model checkpoint")

# Checkpoint Transfer

In [ ]:
print("transfering the checkpoint")

P_PATH = '/kaggle/input/pwc-no-attention-checkpoint/pytorch/v2/1/PwC_DAE_BERT_SMOOTH_64_noATT_V2.pth'

if os.path.exists(P_PATH):
    checkpoint = torch.load(P_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

epoch += 1

print("checkpoint transfered")

# Checkpoint Transfer Ends

In [ ]:
print("incorporating best validation checkpoint")

# if os.path.exists(VAL_BEST_PATH):
#     checkpoint = torch.load(VAL_BEST_PATH)
#     val_loss_checkpoint = checkpoint['val_loss']

# if epoch == 0:
#     best_val_loss = 10e9
# else:
#     best_val_loss = 5.348159131796463

best_val_loss = 7.62133864066779
# best_val_loss = 10e9

print("incorporated best validation checkpoint")

In [ ]:
accumulation_steps = 4  # Number of mini-batches to accumulate gradients

print(f"{'='*25}training started{'='*25}")

best_epoch = 0

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=5e-5)

for epoch in range(epoch, N_EPOCHS):

    start_time = timer()

    model.train()
    epoch_loss = 0

    optimizer.zero_grad()  # Initialize the gradients

    for i, batch in enumerate(tqdm(train_data_loader)):
        src = batch["source_ids"].to(device)
        trg = batch["target_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        
        loss = criterion(output, trg)
        loss = loss / accumulation_steps  # Scale the loss
        
        loss.backward()  # Accumulate gradients
        
        if (i + 1) % accumulation_steps == 0:  # Update model weights after every 'accumulation_steps' mini-batches
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            optimizer.zero_grad()  # Reset gradients for the next accumulation
        
        epoch_loss += loss.item() * accumulation_steps  # Accumulate the true loss (unscaled)

    epoch_loss = epoch_loss / len(train_data_loader)

    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )

    end_time = timer()

    print(f"Loss = {loss}")
    print(f"Epoch Loss = {epoch_loss}")

    print(f"Epoch: {epoch}, Train loss: {epoch_loss}, Val loss: {valid_loss}, Epoch time = {(end_time - start_time):.3f}s")
    print()

    # ===========================================Model Saving====================================
    if epoch_loss < loss:
        loss = epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, BEST_PATH)
        best_epoch = epoch
        print(f"{'-'*50}\nModel Saved at {BEST_PATH}\n{'-'*50}\n")

    else:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, PATH)
        print(f"{'-'*50}\nModel Saved at {PATH}\n{'-'*50}\n")

    # ========================================Model Saved========================================

    #====================================Best Validation Saved===================================
    if best_val_loss > valid_loss:

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
            'val_loss': valid_loss,
        }, VAL_BEST_PATH)
        best_val_loss = valid_loss
        print(f"{'-'*50}\nModel Saved at {VAL_BEST_PATH}\n{'-'*50}\n")

    #     print(f"best validation loss: {best_val_loss}\nvalidation loss: {valid_loss}\n")

    #====================================Best Validation Saved==================================

    scheduler.step()

    for param_group in optimizer.param_groups:
        print(f"Epoch {epoch + 1} Learning Rate: {param_group['lr']}")

    print(f"{'-'*50}\nThe Best Epoch Number: {best_epoch}\n{'-'*50}\n")

print(f"{'='*25}training ended{'='*25}")

In [ ]:
print("incorporating latest model checkpoint")

PATH = '/kaggle/input/pwc-no-attention-checkpoint/pytorch/evaluation/1/PwC_DAE_BERT_SMOOTH_VAL_BEST_PATH_noATT_64_V3.pth'

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated latest model checkpoint")

In [ ]:
test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

In [ ]:
def translate_sentence(
    sentence,
    model,
    tgt_nlp,
    src_nlp,
    target_vocab,
    source_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=1000,
):
    model.eval()
    with torch.no_grad():
        # Tokenize the input sentence
        if isinstance(sentence, str):
            src_tokens = [token for token in src_nlp.tokenize(sentence)]
        else:
            src_tokens = [token for token in sentence]

        # Add <sos> and <eos> tokens to the source sentence
        src_tokens = [sos_token] + src_tokens + [eos_token]
        ids = source_vocab.lookup_indices(src_tokens)

        # Convert to tensor and send to the device
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)

        # Pass through the encoder to get the encoder hidden states
        encoder_outputs, hidden = model.encoder(tensor)

        # Start decoding with <sos> token
        inputs = target_vocab.lookup_indices([sos_token])
        for i in range(max_output_length):
            # Convert the last predicted token to tensor and pass it to the decoder
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden = model.decoder(inputs_tensor, hidden)

            # Get the predicted token (argmax of output)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)

            # Stop if <eos> token is predicted
            if predicted_token == target_vocab[eos_token]:
                break

        # Convert predicted token indices back to words
        tgt_tokens = target_vocab.lookup_tokens(inputs)

    # Return the target tokens and source tokens
    return tgt_tokens, src_tokens

In [ ]:
def plot_attention(sentence, translation, attention):
    fig, ax = plt.subplots(figsize=(10, 10))
    attention = attention.squeeze(1).numpy()
    cax = ax.matshow(attention, cmap="bone")
    ax.set_xticks(ticks=np.arange(len(sentence)), labels=sentence, rotation=90, size=15)
    translation = translation[1:]
    ax.set_yticks(ticks=np.arange(len(translation)), labels=translation, size=15)
    plt.show()
    plt.close()

In [ ]:
sentence = test_data[0]["source"]
expected_translation = test_data[0]["target"]

sentence, expected_translation

In [ ]:
translation, sentence_tokens = translate_sentence(
    sentence,
    model,
    tgt_nlp,
    src_nlp,
    target_vocab,
    source_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [ ]:
translation

In [ ]:
sentence_tokens

In [ ]:
# plot_attention(sentence_tokens, translation, attention)

In [ ]:
sentence = "Therefore, it's important for the government to develop policies."

In [ ]:
translation, sentence_tokens = translate_sentence(
    sentence,
    model,
    tgt_nlp,
    src_nlp,
    target_vocab,
    source_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [ ]:
translation

# **Evaluation**

In [ ]:
class TextDataset(Dataset):

    def __init__(self, source_sentences, target_sentences):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, idx):
        return self.source_sentences[idx], self.target_sentences[idx]

In [ ]:
eval_dataset = TextDataset(test_source_sentences, test_target_sentences)

In [ ]:
eval_dataset.__len__(), eval_dataset.__getitem__(0)

In [ ]:
!pip install sacrebleu
!pip install evaluate
!pip install bert_score evaluate
!pip install torchmetrics
!pip install rouge_score

In [ ]:
import datasets
from evaluate import load
import nltk
from nltk.translate.bleu_score import sentence_bleu
import sacrebleu
import evaluate


sacrebleu = evaluate.load("sacrebleu")
bertscore = load("bertscore")
bleu = evaluate.load("bleu")
rouge = evaluate.load('rouge')

In [ ]:
def calculate_wer(reference, candidate):
    reference = reference.split()
    candidate = candidate.split()

    # Create a matrix to store distances
    distance_matrix = np.zeros((len(reference) + 1) * (len(candidate) + 1), dtype=int).reshape((len(reference) + 1, len(candidate) + 1))

    for i in range(len(reference) + 1):
        for j in range(len(candidate) + 1):
            if i == 0:
                distance_matrix[i][j] = j
            elif j == 0:
                distance_matrix[i][j] = i
            elif reference[i - 1] == candidate[j - 1]:
                distance_matrix[i][j] = distance_matrix[i - 1][j - 1]
            else:
                distance_matrix[i][j] = 1 + min(distance_matrix[i][j - 1],      # Insert
                                               distance_matrix[i - 1][j],      # Remove
                                               distance_matrix[i - 1][j - 1])  # Replace

    return distance_matrix[len(reference)][len(candidate)]

In [ ]:
import re

def autoregressive_decoding(sentence):
    translation, sentence_tokens = translate_sentence(
      sentence,
      model,
      tgt_nlp,
      src_nlp,
      target_vocab,
      source_vocab,
      lower,
      sos_token,
      eos_token,
      device,
    )

    # Filter out unwanted tokens
    filtered_translation = [word for word in translation if word not in ('<sos>', '<eos>', '<unk>')]

    # Join words into a sentence
    sentence = ' '.join(filtered_translation).replace(" ##", "")

    # Remove spaces before punctuation (comma, period, etc.)
    sentence = re.sub(r'\s+([.,!?;:])', r'\1', sentence)

    # Remove spaces around single and double quotes
    sentence = re.sub(r"\s*(['\"])\s*", r'\1', sentence)

    # Remove spaces around apostrophes in contractions
    sentence = re.sub(r"\s*'\s*(\w+)", r"'\1", sentence)

    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    # Remove spaces after opening and before closing parentheses/brackets
    sentence = re.sub(r'\s*\(\s*', '(', sentence)
    sentence = re.sub(r'\s*\)\s*', ')', sentence)

    # Ensure proper ellipsis formatting (already in the original code)
    sentence = re.sub(r'\s*\.\s*\.\s*\.\s*', '...', sentence)

    # Remove spaces around hyphens or dashes
    sentence = re.sub(r'\s*-\s*', '-', sentence)

    # Place punctuation inside quotation marks (handle most common cases)
    sentence = re.sub(r'([.,!?;:])\s*([\'"])', r'\2\1', sentence)

    # Remove spaces around "@" for email addresses and Twitter handles
    sentence = re.sub(r'\s*@\s*', '@', sentence)

    # Remove spaces around slashes (e.g., in URLs or subreddit names)
    sentence = re.sub(r'\s*/\s*', '/', sentence)

    # Remove spaces around currency symbols and numbers
    sentence = re.sub(r'\s*([$€£])\s*([0-9]+)', r' \1\2', sentence)
    sentence = re.sub(r'([0-9]+)\s*([,])\s*([0-9]+)', r'\1\2\3', sentence)
    sentence = re.sub(r'([0-9]+)\s*([%])', r'\1\2', sentence)

    # Remove spaces within URLs
    sentence = re.sub(r'\s*(http://|https://|www\.)\s*', r'\1', sentence)

    # Remove spaces in dates and times
    sentence = re.sub(r'\s*/\s*', '/', sentence)  # For dates with slashes
    sentence = re.sub(r'\s*:\s*', ':', sentence)  # For times or other similar cases

    # Remove spaces around mathematical operators
    sentence = re.sub(r'\s*([+\-*/=])\s*', r'\1', sentence)

    # Remove spaces around file extensions
    sentence = re.sub(r'\s+(\.[a-zA-Z0-9]+)', r'\1', sentence)

    # Remove spaces around code-like markers (assuming `code` is delimited by backticks)
    sentence = re.sub(r'\s*(```)\s*', r'\1', sentence)

    # --- New logic for removing spaces after dots ---

    # 1. Remove spaces after a dot when followed by a number (for decimals, currency, dates)
    sentence = re.sub(r'([0-9])\.\s+([0-9])', r'\1.\2', sentence)

    # 2. Remove spaces after a dot when followed by a letter (for abbreviations like U.S.A)
    sentence = re.sub(r'([A-Za-z])\.\s+([A-Za-z])', r'\1.\2', sentence)

    # 3. Remove spaces after a dot when followed by punctuation (e.g., ". , ;")
    sentence = re.sub(r'\.\s+([.,!?;:])', r'.\1', sentence)

    # Ensure there's a space before an opening parenthesis if preceded by a word
    sentence = re.sub(r'(\w)\(', r'\1 (', sentence)

    # Ensure there's a space after a closing parenthesis if followed by a word
    sentence = re.sub(r'\)(\w)', r') \1', sentence)

    # Ensure there's a space after a closing parenthesis if followed by any symbol (%, $, etc.)
    sentence = re.sub(r'\)([^\w\s])', r') \1', sentence)

    # Remove spaces between closing parenthesis and punctuation
    sentence = re.sub(r'\)\s+([,.;!?])', r')\1', sentence)

    # 4. Handle cases where a dot is followed by multiple dots but not an ellipsis (e.g., "U.S. ...")
    sentence = re.sub(r'\.\s+\.', r'..', sentence)

    return sentence

In [ ]:
import re
from tqdm import tqdm

print("evaluation begins!")

total_bleu_score = 0
total_sacre_bleu_score = 0
total_wer_score = 0
total_precision = 0
total_recall = 0
total_f1 = 0

# Initialize variables for Rouge scores
total_rouge1_score = 0
total_rouge2_score = 0
total_rougeL_score = 0

# Create lists to store individual scores for each sentence
bleu_scores = []
sacre_bleu_scores = []
wer_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Create list for storing DAE generated output and target text
dae_generated_output = []
target_text_list = []

# Create lists to store Rouge scores for each sentence
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

# sacrebleu_hf = load_metric("sacrebleu")
# bertscore = load("bertscore")

for (source_sentence, target_sentence) in tqdm(eval_dataset):

    output_text = autoregressive_decoding(source_sentence)
    target_sentence = target_sentence.lower()
    print(f"\nDAE output: {output_text}\ntarget sentence: {target_sentence}\n")

    # Calculate BLEU score using SacreBLEU
    temp_bleu_score = bleu.compute(predictions=[output_text], references=[[target_sentence]], max_order=2)
    bleu_score = temp_bleu_score["bleu"]

    if bleu_score < 1 or bleu_score < 1.0:
        dae_generated_output.append(output_text)
        target_text_list.append(target_sentence + " BLEU SCORE" + str(bleu_score))
      # print(f"\nDAE output: {output_text}\ntarget sentence: {target_sentence}\n")
      # print(f"\nBLEU Score: {bleu_score}\n")

    # Calculate Rouge scores
    temp_rouge_score = rouge.compute(predictions=[output_text], references=[target_sentence])
    rouge1_score = temp_rouge_score["rouge1"]
    rouge2_score = temp_rouge_score["rouge2"]
    rougeL_score = temp_rouge_score["rougeL"]

    # Calculate SacreBLEU and WER scores
    sacre_bleu_score = sacrebleu.compute(predictions=[output_text], references=[[target_sentence]])
    sacre_bleu_score = sacre_bleu_score["score"]
    wer_score = calculate_wer(target_sentence, output_text)

    # Calculate BERTScore metrics
    results = bertscore.compute(predictions=[output_text], references=[target_sentence], lang="en")

    precision_score = results["precision"]
    recall_score = results["recall"]
    f1_score = results["f1"]

    if isinstance(precision_score, list):
        precision_score = sum(precision_score) / len(precision_score)
    if isinstance(recall_score, list):
        recall_score = sum(recall_score) / len(recall_score)
    if isinstance(f1_score, list):
        f1_score = sum(f1_score) / len(f1_score)

    # Accumulate scores
    total_bleu_score += bleu_score

    total_rouge1_score += rouge1_score
    total_rouge2_score += rouge2_score
    total_rougeL_score += rougeL_score

    total_sacre_bleu_score += sacre_bleu_score
    total_wer_score += wer_score

    total_precision += precision_score
    total_recall += recall_score
    total_f1 += f1_score

    # Store individual scores for each sentence
    bleu_scores.append(bleu_score)
    rouge1_scores.append(rouge1_score)
    rouge2_scores.append(rouge2_score)
    rougeL_scores.append(rougeL_score)

    sacre_bleu_scores.append(sacre_bleu_score)
    wer_scores.append(wer_score)

    precision_scores.append(precision_score)
    recall_scores.append(recall_score)
    f1_scores.append(f1_score)


# Calculate average scores
avg_bleu_score = total_bleu_score / len(eval_dataset)
avg_sacre_bleu_score = total_sacre_bleu_score / len(eval_dataset)
avg_wer_score = total_wer_score / len(eval_dataset)

avg_rouge1_score = total_rouge1_score / len(eval_dataset)
avg_rouge2_score = total_rouge2_score / len(eval_dataset)
avg_rougeL_score = total_rougeL_score / len(eval_dataset)

avg_PR_score = total_precision / len(eval_dataset)
avg_RE_score = total_recall / len(eval_dataset)
avg_F1_score = total_f1 / len(eval_dataset)

# Print average scores
print()
print(f"Average BLEU Score: {(avg_bleu_score * 100):.4f}")
print(f"Average SacreBLEU Score: {avg_sacre_bleu_score:.4f}")
print(f"Average Word Error Rate (WER): {avg_wer_score}")

print(f"Average Rouge-1 Score: {avg_rouge1_score}")
print(f"Average Rouge-2 Score: {avg_rouge2_score}")
print(f"Average Rouge-L Score: {avg_rougeL_score}")

print(f"Average Precision: {avg_PR_score}")
print(f"Average Recall: {avg_RE_score}")
print(f"Average F1-Score: {avg_F1_score}")

print("evaluation ends!")